# Task 3 — Manual (Human) Evaluation

Load the `assignment_01.xlsx` produced in Task 2, add a cost column, manually rate 10–15 products, compute final scores, and analyze which criteria performed best and worst.

In [32]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, str(Path.cwd().parent))
load_dotenv(Path.cwd().parent / ".env")

from shared.config import GENERATION_MODEL, OUTPUT_EXCEL
from shared.constants import CRITERIA, final_score
from shared.utils import calculate_cost, rate_cost, rate_latency, rate_length

In [33]:
df = pd.read_excel(OUTPUT_EXCEL)
print(f"{len(df)} products loaded")
df.head(3)

50 products loaded


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score,cost_usd
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Experience the power and performance of the Ap...,1454.7,303,98,good,good,good,good,good,good,good,pass,0.000080
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the stunning 200MP c...,1058.3,303,109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000082
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture stunning moments with the Google Pixel...,937.5,300,96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000079


## 1. Cost Calculation

Convert `input_tokens` and `output_tokens` to USD using the per-1K-token pricing for Gemma-2-9b-it. Note: input and output tokens can be priced differently.

In [34]:
df["cost_usd"] = df.apply(
    lambda row: calculate_cost(GENERATION_MODEL, row["input_tokens"], row["output_tokens"]),
    axis=1,
)

print(f"Total cost: ${df['cost_usd'].sum():.6f}")
print(f"Avg cost per call: ${df['cost_usd'].mean():.6f}")
df[["product_name", "input_tokens", "output_tokens", "cost_usd"]].head()

Total cost: $0.003980
Avg cost per call: $0.000080


,product_name,input_tokens,output_tokens,cost_usd
0,Apple iPhone 15 Pro,303,98,0.000080
1,Samsung Galaxy S24 Ultra,303,109,0.000082
2,Google Pixel 8 Pro,300,96,0.000079
3,Sony WH‑1000XM5 Headphones,300,95,0.000079
4,Bose QuietComfort Ultra Earbuds,289,99,0.000078


## 2. Rate Each Criterion (10–15 products)

15 products are selected (evenly spaced across the dataset) for manual rating of each criterion as good / ok / bad according to the rubric from Task 1.

Latency and Cost are rated programmatically based on the thresholds in the rubric.

In [35]:
# Select 15 evenly spaced indices for manual evaluation
import numpy as np

EVAL_COUNT = 15
eval_indices = np.linspace(0, len(df) - 1, EVAL_COUNT, dtype=int).tolist()
print(f"Products selected for evaluation (indices): {eval_indices}")
print()
for i in eval_indices:
    row = df.iloc[i]
    word_count = len(row["generated_description"].split())
    print(f"--- [{i}] {row['product_name']} ({word_count} words) ---")
    print(row["generated_description"])
    print()

Products selected for evaluation (indices): [0, 3, 7, 10, 14, 17, 21, 24, 28, 31, 35, 38, 42, 45, 49]

--- [0] Apple iPhone 15 Pro (66 words) ---
Experience the power and performance of the Apple iPhone 15 Pro.  Fueled by the groundbreaking A17 Pro chip, this compact powerhouse delivers a seamless user experience with its 120Hz ProMotion display.  The durable titanium frame and Ceramic Shield glass ensure lasting protection, while USB‑C fast charging keeps you connected. Backed by a 1-year limited warranty, the iPhone 15 Pro is ready to elevate your mobile experience.

--- [3] Sony WH‑1000XM5 Headphones (62 words) ---
Escape into your music with the Sony WH-1000XM5 Headphones. Enjoy immersive sound and crystal-clear calls thanks to advanced active noise cancelling technology.  With a comfortable, large design and plush synthetic leather earcups, you can wear these headphones for hours on end.  The impressive 30-hour battery life and Bluetooth 5.2 connectivity ensure uninterrupted liste

In [36]:
# Keys: product index -> {criterion: rating}
# Only Fluency, Grammar, Tone, Grounding need manual judgment.
# Length, Latency, Cost are computed automatically.

manual_ratings: dict[int, dict[str, str]] = {
    0:  {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    3:  {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    7:  {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "ok"},
    10: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    14: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    17: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    21: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    24: {"Fluency": "good", "Grammar": "good", "Tone": "ok",   "Grounding": "good"},
    28: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    31: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    35: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    38: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    42: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "ok"},
    45: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
    49: {"Fluency": "good", "Grammar": "good", "Tone": "good", "Grounding": "good"},
}

assert len(manual_ratings) >= 10, (
    f"Rate at least 10 products. Currently rated: {len(manual_ratings)}"
)

## 3. Apply Final Score

In [37]:
for idx, human_scores in manual_ratings.items():
    row = df.iloc[idx]
    full_ratings = {
        **human_scores,
        "Length": rate_length(row["generated_description"]),
        "Latency": rate_latency(row["latency_ms"]),
        "Cost": rate_cost(row["cost_usd"]),
    }
    for criterion, rating in full_ratings.items():
        df.at[idx, criterion] = rating
    df.at[idx, "final_score"] = final_score(full_ratings)

evaluated = df[df["final_score"].isin(["pass", "fail"])]
print(f"Evaluated {len(evaluated)} products")
evaluated[["product_name"] + CRITERIA + ["final_score"]]

Evaluated 15 products


,product_name,Fluency,Grammar,Tone,Length,Grounding,Latency,Cost,final_score
0,Apple iPhone 15 Pro,good,good,good,good,good,good,good,pass
3,Sony WH‑1000XM5 Headphones,good,good,good,good,good,good,good,pass
7,Apple MacBook Air 13″ (M3),good,good,good,good,ok,good,good,pass
10,Fitbit Charge 6,good,good,good,good,good,good,good,pass
14,PlayStation 5 Slim,good,good,good,good,good,good,good,pass
17,Keurig K‑Supreme Plus Smart,good,good,good,good,good,good,good,pass
21,Yeti Rambler 20 oz Tumbler,good,good,good,good,good,good,good,pass
24,Contigo Autoseal West Loop 16 oz,good,good,ok,good,good,good,good,pass
28,Patagonia Black Hole 32 L Backpack,good,good,good,good,good,good,good,pass
31,LEGO Icons Botanical Orchid 10311,good,good,good,good,good,good,good,pass


In [38]:
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Saved to {OUTPUT_EXCEL}")

Saved to /Users/nisim/Desktop/Nebuis Academy/Lesson-1-homeowrk/task_3/assignment_01.xlsx


## 4. Baseline Analysis

In [39]:
rating_to_num = {"good": 2, "ok": 1, "bad": 0}

print("=== Pass / Fail Summary ===")
print(evaluated["final_score"].value_counts())
print(f"Pass rate: {(evaluated['final_score'] == 'pass').mean():.0%}")
print()

print("=== Rating Distribution per Criterion ===")
for c in CRITERIA:
    counts = evaluated[c].value_counts()
    print(f"\n{c}:")
    for rating in ["good", "ok", "bad"]:
        print(f"  {rating}: {counts.get(rating, 0)}")

print()
print("=== Average Score per Criterion (good=2, ok=1, bad=0) ===")
avg_scores = {}
for c in CRITERIA:
    numeric = evaluated[c].map(rating_to_num)
    avg_scores[c] = numeric.mean()

sorted_criteria = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
for c, score in sorted_criteria:
    print(f"  {c}: {score:.2f}")

print(f"\nBest performing:  {sorted_criteria[0][0]} ({sorted_criteria[0][1]:.2f})")
print(f"Worst performing: {sorted_criteria[-1][0]} ({sorted_criteria[-1][1]:.2f})")

=== Pass / Fail Summary ===
final_score
pass    15
Name: count, dtype: int64
Pass rate: 100%

=== Rating Distribution per Criterion ===

Fluency:
  good: 15
  ok: 0
  bad: 0

Grammar:
  good: 15
  ok: 0
  bad: 0

Tone:
  good: 14
  ok: 1
  bad: 0

Length:
  good: 15
  ok: 0
  bad: 0

Grounding:
  good: 13
  ok: 2
  bad: 0

Latency:
  good: 15
  ok: 0
  bad: 0

Cost:
  good: 15
  ok: 0
  bad: 0

=== Average Score per Criterion (good=2, ok=1, bad=0) ===
  Fluency: 2.00
  Grammar: 2.00
  Length: 2.00
  Latency: 2.00
  Cost: 2.00
  Tone: 1.93
  Grounding: 1.87

Best performing:  Fluency (2.00)
Worst performing: Grounding (1.87)


## Baseline Analysis Summary

- **Best criteria:** Fluency, Grammar, Latency, Cost — all scored a perfect 2.00 across 15 evaluated products. The model consistently produces fluent, grammatically correct text with fast response times and low cost.
- **Worst criterion:** Grounding (1.87) — two descriptions contained slight exaggerations or marketing claims not directly stated in the product data (e.g., "groundbreaking" for MacBook M3, "unparalleled speed" for Razer mouse).
- **Pass rate:** 100% (15/15 passed)
- **Key observations:** The Gemma-2-9b-it model performs well overall on this task. Nearly all descriptions land in the 50–90 word range, the tone is consistently professional, and grammar is clean. The main weakness is occasional grounding issues where the model adds subjective superlatives beyond what the product data states.
- **Improvement strategy for Task 4:** Focus on tightening grounding by adding a few-shot example that demonstrates sticking strictly to provided information. Also try lowering temperature to reduce creative embellishment.